# FineTuneOps — Task 4: QLoRA Supervised Fine-Tuning (Cloud GPU)
### Domain-Specialized LLM Adaptation for DevOps Incident Diagnosis

This notebook provides the primary cloud GPU training workflow for **FineTuneOps**.
It fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` using 4-bit NormalFloat (NF4) QLoRA on the official **Cloud-OpsBench** benchmark dataset (514 train cases, 92 validation cases).

**Architectural & Benchmark Constraints:**
1. **Exact Base Model**: `Qwen/Qwen2.5-Coder-1.5B-Instruct` (Revision: `2e1fd397ee46e1388853d2af2c993145b0f1098a`).
2. **QLoRA Hyperparameters**: LoRA rank $r=16$, alpha $\alpha=32$, dropout $0.05$, targeting all 7 attention and MLP linear projections (`q, k, v, o, gate, up, down`).
3. **Long Context Window**: `max_length = 3072` tokens to preserve the extensive diagnostic trace without truncation.
4. **Strict Benchmark Isolation**: The 148 held-out test cases (`test.jsonl`) are strictly isolated and never accessed during training.
5. **Any-GPU Support**: Dynamically detects available GPU (T4, V100, A100, L4) and adjusts compute dtypes (`bfloat16` or `float16`).

## 1. Hardware Detection & Dynamic Environment Assessment

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "CUDA GPU is required! Please go to Runtime -> Change runtime type -> select T4, A100, or L4 GPU."

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
bf16_supported = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

print(f"Detected GPU: {gpu_name}")
print(f"Total Physical VRAM: {total_vram_gb:.2f} GB")
print(f"bfloat16 Supported: {bf16_supported}")
print(f"Selected Compute Dtype: {compute_dtype}")

## 2. Install Pinned Dependencies

In [ ]:
!pip install -q "transformers>=4.40.0" "datasets>=2.18.0" "peft>=0.10.0" "trl>=0.8.6" "accelerate>=0.29.0" "bitsandbytes>=0.43.0" "scikit-learn" "matplotlib"

## 3. Clone Repository or Prepare Dataset Splits
Ensure `ml/data/splits/train.jsonl` and `val.jsonl` are present.

In [ ]:
import os, json
from pathlib import Path

# If running standalone, clone or create directories
if not Path("ml/data/splits/train.jsonl").exists():
    print("Repository not cloned yet. If you have a GitHub repository, run:")
    print("!git clone https://github.com/<your-user>/FineTuneOps.git && %cd FineTuneOps")
    print("Otherwise, upload train.jsonl and val.jsonl using the upload widget below.")
    from google.colab import files
    os.makedirs("ml/data/splits", exist_ok=True)
    uploaded = files.upload()
    for fname, data in uploaded.items():
        with open(os.path.join("ml/data/splits", fname), "wb") as f:
            f.write(data)

# Audit dataset files
train_path = Path("ml/data/splits/train.jsonl")
val_path = Path("ml/data/splits/val.jsonl")
assert train_path.exists(), "ml/data/splits/train.jsonl is missing!"
assert val_path.exists(), "ml/data/splits/val.jsonl is missing!"

train_cases = [json.loads(l) for l in open(train_path, encoding='utf-8') if l.strip()]
val_cases = [json.loads(l) for l in open(val_path, encoding='utf-8') if l.strip()]
print(f"Verified dataset splits: Train={len(train_cases)} cases, Val={len(val_cases)} cases.")
assert len(train_cases) == 514, f"Expected 514 train cases, found {len(train_cases)}"
assert len(val_cases) == 92, f"Expected 92 validation cases, found {len(val_cases)}"

## 4. Run QLoRA Supervised Fine-Tuning (SFT)
Fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with 4-bit NF4 quantization, LoRA rank 16, and sequence length 3072.

In [ ]:
import time, json
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_DIR = Path("models/finetuned/adapter")
BEST_DIR = Path("models/finetuned/adapter/best")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR.mkdir(parents=True, exist_ok=True)

# 1. 4-bit NormalFloat (NF4) Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# 2. Tokenizer
print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Model Loading with 4-bit Quantization
print(f"Loading base model in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# 4. PEFT LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
trainable_p, total_p = model.get_nb_trainable_parameters()
print(f"LoRA Parameters: {trainable_p:,} / {total_p:,} ({100*trainable_p/total_p:.4f}% trainable)")

# 5. Dataset Formatting
def format_chat_item(item):
    return {"text": tokenizer.apply_chat_template(item["messages"], tokenize=False)}

train_ds = Dataset.from_list([format_chat_item(x) for x in train_cases])
val_ds = Dataset.from_list([format_chat_item(x) for x in val_cases])

# 6. Loss Monitoring Callback
class MetricTrackerCallback(TrainerCallback):
    def __init__(self):
        self.step_history = []
        self.eval_history = []
        self.peak_vram_mb = 0.0
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        cur_vram = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
        self.peak_vram_mb = max(self.peak_vram_mb, cur_vram)
        if "loss" in logs:
            self.step_history.append({"step": state.global_step, "epoch": state.epoch, "train_loss": float(logs["loss"]), "peak_vram_mb": round(cur_vram, 2)})
        if "eval_loss" in logs:
            self.eval_history.append({"step": state.global_step, "epoch": state.epoch, "eval_loss": float(logs["eval_loss"])})

tracker = MetricTrackerCallback()

# 7. SFT Configuration
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # Effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    num_train_epochs=3,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    seed=42,
    max_length=3072,
    dataset_text_field="text",
    report_to=["none"],
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    processing_class=tokenizer,
    callbacks=[tracker],
)

start_t = time.time()
print("Starting QLoRA Supervised Fine-Tuning (3 epochs)... ")
train_res = trainer.train()
duration_sec = round(time.time() - start_t, 2)
print(f"Training finished in {duration_sec:.2f}s ({duration_sec/60:.2f} minutes)!")

# 8. Save Final and Best Adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.model.save_pretrained(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print("Adapters successfully saved to:", OUTPUT_DIR, "and", BEST_DIR)

## 5. Loss Curve Visualization & Overfitting Audit

In [ ]:
os.makedirs("experiments/reports", exist_ok=True)
os.makedirs("experiments/results", exist_ok=True)

plt.figure(figsize=(9, 5))
steps = [x["step"] for x in tracker.step_history if "train_loss" in x]
losses = [x["train_loss"] for x in tracker.step_history if "train_loss" in x]
if steps and losses:
    plt.plot(steps, losses, label="Training Loss (Step)", color="#2563eb", alpha=0.7)
    if len(losses) >= 5:
        ma = np.convolve(losses, np.ones(5)/5, mode="valid")
        plt.plot(steps[2:len(ma)+2], ma, label="Training Loss (5-step MA)", color="#1d4ed8", linewidth=2)

if tracker.eval_history:
    e_steps = [x["step"] for x in tracker.eval_history]
    e_losses = [x["eval_loss"] for x in tracker.eval_history]
    plt.scatter(e_steps, e_losses, color="#dc2626", s=60, zorder=5)
    plt.plot(e_steps, e_losses, label="Validation Loss (Epoch)", color="#dc2626", linestyle="--", linewidth=2)

plt.title("FineTuneOps — QLoRA Training and Validation Loss Curve", fontsize=13, fontweight="bold")
plt.xlabel("Training Step", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("experiments/reports/training_loss_curve.png", dpi=200)
plt.show()

# Overfitting check
final_train = tracker.step_history[-1]["train_loss"] if tracker.step_history else 0.0
final_val = tracker.eval_history[-1]["eval_loss"] if tracker.eval_history else 0.0
print(f"Final Training Loss: {final_train:.4f}")
print(f"Final Validation Loss: {final_val:.4f}")
print(f"Peak GPU Memory: {tracker.peak_vram_mb:.2f} MB")

## 6. Verify Adapter Loading & Export Checkpoint Metrics

In [ ]:
# Verify adapter loads cleanly into base model
print("Verifying adapter artifact integrity...")
test_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
verified_model = PeftModel.from_pretrained(test_base, str(OUTPUT_DIR))
print("Verification SUCCESSFUL: Adapter loads cleanly into foundation model!")
del test_base, verified_model
torch.cuda.empty_cache()

# Export JSON metrics
metrics_payload = {
    "base_model": MODEL_ID,
    "base_model_revision": "2e1fd397ee46e1388853d2af2c993145b0f1098a",
    "hardware": {"device": gpu_name, "vram_total_gb": round(total_vram_gb, 2), "peak_vram_mb": round(tracker.peak_vram_mb, 2)},
    "parameter_counts": {"trainable_parameters": int(trainable_p), "total_parameters": int(total_p), "trainable_percentage": round(100*trainable_p/total_p, 4)},
    "training_statistics": {
        "num_epochs": 3,
        "total_train_samples": 514,
        "total_val_samples": 92,
        "max_sequence_length": 3072,
        "final_train_loss": round(final_train, 4),
        "final_validation_loss": round(final_val, 4),
        "best_validation_loss": min([x["eval_loss"] for x in tracker.eval_history]) if tracker.eval_history else final_val,
        "training_duration_seconds": duration_sec,
    },
    "step_loss_trajectory": tracker.step_history,
    "epoch_validation_losses": tracker.eval_history,
}
with open("experiments/results/training_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print("Saved metrics to experiments/results/training_metrics.json")

## 7. Package Adapter for Local FineTuneOps Deployment

In [ ]:
!zip -r adapter.zip models/finetuned/adapter experiments/results/training_metrics.json experiments/reports/training_loss_curve.png
from google.colab import files
print("Downloading adapter package...")
files.download("adapter.zip")
# Extract this adapter.zip directly into your local FineTuneOps workspace!